In [2]:
"""
Real Data Fetcher — HDB, TCB, TPB (2019-01-01 → 2025-12-31)
=============================================================
Produces ONE unified CSV: stock_dataset_full.csv
Compatible with ARIMA, LSTM, and TFT — all columns in one file.

Data sources
------------
  Stock prices  : vnstock (TCBS)  →  fallback: yfinance (HDB.VN, TCB.VN, TPB.VN)
  VN-Index      : vnstock (TCBS)  →  fallback: yfinance (^VNINDEX.VN)
  USD/VND rate  : yfinance (USDVND=X)
  Gold price    : yfinance (GC=F USD/oz)  →  converted to approximate VND
  SBV rate      : hardcoded known policy decisions (publicly available)

Install
-------
  pip install vnstock yfinance pandas numpy

Usage
-----
  python fetch_stock_dataset.py
  # Outputs: stock_dataset_full.csv  +  stock_dataset_train.csv  +  stock_dataset_test.csv

Column guide (single file, use what each model needs)
------------------------------------------------------
  ARIMA  → date, ticker, close
  LSTM   → date, ticker, open, high, low, close, volume, [macro cols], [technical cols]
  TFT    → ALL columns (adds: static_*, cal_*, cross_lag_*)
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import time
import sys

START = "2019-01-01"
END   = "2025-12-31"
SPLIT = "2025-10-01"          # train / test boundary

TICKERS = ["HDB", "TCB", "TPB"]

# ─────────────────────────────────────────────────────────────────
# SECTION 1 — STOCK PRICE FETCH
# ─────────────────────────────────────────────────────────────────

def fetch_vnstock(symbol, start, end):
    """Try vnstock v3 (modern) then v2 (legacy) API."""
    # --- vnstock v3 (pip install vnstock >= 3.0) ---
    try:
        from vnstock import Quote
        q = Quote(symbol=symbol, source="TCBS")
        df = q.history(start=start, end=end, interval="1D")
        df = df.rename(columns=str.lower)
        # vnstock v3 returns columns: time, open, high, low, close, volume
        date_col = next((c for c in df.columns if c in ["time", "date", "tradingdate"]), None)
        if date_col:
            df = df.rename(columns={date_col: "date"})
        df["date"] = pd.to_datetime(df["date"])
        df = df[["date", "open", "high", "low", "close", "volume"]].copy()
        df["ticker"] = symbol
        print(f"  [vnstock v3] {symbol}: {len(df)} rows")
        return df
    except Exception as e1:
        pass

    # --- vnstock v2 / legacy (pip install vnstock == 0.x) ---
    try:
        from vnstock import stock_historical_data
        df = stock_historical_data(symbol, start, end)
        df = df.rename(columns=str.lower)
        df = df.rename(columns={"tradingdate": "date", "tradingDate": "date"})
        df["date"] = pd.to_datetime(df["date"])
        df = df[["date", "open", "high", "low", "close", "volume"]].copy()
        df["ticker"] = symbol
        print(f"  [vnstock v2] {symbol}: {len(df)} rows")
        return df
    except Exception as e2:
        return None


def fetch_yfinance_stock(symbol, start, end):
    """Fallback: fetch from Yahoo Finance (HDB.VN format)."""
    try:
        import yfinance as yf
        yf_sym = f"{symbol}.VN"
        raw = yf.download(yf_sym, start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            return None
        raw = raw.reset_index()
        raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                       for c in raw.columns]
        raw = raw.rename(columns={"date": "date"})
        raw["date"] = pd.to_datetime(raw["date"])
        df = raw[["date", "open", "high", "low", "close", "volume"]].copy()
        df["ticker"] = symbol
        # yfinance returns VND already for .VN tickers
        print(f"  [yfinance]  {symbol}.VN: {len(df)} rows")
        return df
    except Exception as e:
        print(f"  [yfinance]  {symbol}.VN failed: {e}")
        return None


def fetch_stock(symbol, start, end):
    df = fetch_vnstock(symbol, start, end)
    if df is None or len(df) < 100:
        print(f"  vnstock gave <100 rows for {symbol}, trying yfinance...")
        df = fetch_yfinance_stock(symbol, start, end)
    if df is None:
        raise RuntimeError(f"Could not fetch price data for {symbol}")
    return df.sort_values("date").reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────
# SECTION 2 — VN-INDEX
# ─────────────────────────────────────────────────────────────────

def fetch_vnindex(start, end):
    # Try vnstock first
    try:
        from vnstock import Quote
        q = Quote(symbol="VNINDEX", source="TCBS")
        df = q.history(start=start, end=end, interval="1D")
        df = df.rename(columns=str.lower)
        date_col = next((c for c in df.columns if c in ["time", "date", "tradingdate"]), None)
        if date_col:
            df = df.rename(columns={date_col: "date"})
        df["date"] = pd.to_datetime(df["date"])
        df = df[["date", "close"]].rename(columns={"close": "vni_close"})
        df["vni_change_pct"] = df["vni_close"].pct_change() * 100
        print(f"  [vnstock]   VNINDEX: {len(df)} rows")
        return df
    except Exception:
        pass

    # Fallback: yfinance
    try:
        import yfinance as yf
        raw = yf.download("^VNINDEX.VN", start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            raw = yf.download("^VNINDEX", start=start, end=end, progress=False, auto_adjust=True)
        raw = raw.reset_index()
        raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                       for c in raw.columns]
        df = raw[["date", "close"]].rename(columns={"close": "vni_close"})
        df["date"] = pd.to_datetime(df["date"])
        df["vni_change_pct"] = df["vni_close"].pct_change() * 100
        print(f"  [yfinance]  VNINDEX: {len(df)} rows")
        return df
    except Exception as e:
        print(f"  WARNING: Could not fetch VNINDEX: {e}. Filling with NaN.")
        return None


# ─────────────────────────────────────────────────────────────────
# SECTION 3 — USD/VND EXCHANGE RATE
# ─────────────────────────────────────────────────────────────────

def fetch_usdvnd(start, end):
    try:
        import yfinance as yf
        raw = yf.download("USDVND=X", start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            raise ValueError("Empty USDVND data")
        raw = raw.reset_index()
        raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                       for c in raw.columns]
        df = raw[["date", "close"]].rename(columns={"close": "usd_vnd"})
        df["date"] = pd.to_datetime(df["date"])
        print(f"  [yfinance]  USD/VND: {len(df)} rows")
        return df
    except Exception as e:
        print(f"  WARNING: USD/VND fetch failed ({e}). Using linear interpolation fallback.")
        return None


def usdvnd_fallback(date_range):
    """Approximate USD/VND from known historical values."""
    anchor = {
        "2019-01-01": 23200, "2020-01-01": 23180, "2020-04-01": 23500,
        "2021-01-01": 23100, "2022-01-01": 23100, "2022-10-01": 24800,
        "2023-01-01": 23650, "2024-01-01": 24200, "2024-07-01": 25450,
        "2025-01-01": 25500, "2025-12-31": 25800,
    }
    anchor_df = pd.DataFrame([
        {"date": pd.Timestamp(k), "usd_vnd": v} for k, v in anchor.items()
    ]).sort_values("date")
    full = pd.DataFrame({"date": date_range})
    merged = full.merge(anchor_df, on="date", how="left")
    merged["usd_vnd"] = merged["usd_vnd"].interpolate(method="linear").ffill().bfill()
    return merged


# ─────────────────────────────────────────────────────────────────
# SECTION 4 — GOLD PRICE (USD → approximate VND)
# ─────────────────────────────────────────────────────────────────

def fetch_gold(start, end):
    try:
        import yfinance as yf
        raw = yf.download("GC=F", start=start, end=end, progress=False, auto_adjust=True)
        if raw.empty:
            raise ValueError("Empty gold data")
        raw = raw.reset_index()
        raw.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                       for c in raw.columns]
        df = raw[["date", "close"]].rename(columns={"close": "gold_usd_oz"})
        df["date"] = pd.to_datetime(df["date"])
        print(f"  [yfinance]  Gold (GC=F): {len(df)} rows")
        return df
    except Exception as e:
        print(f"  WARNING: Gold fetch failed ({e}).")
        return None


# ─────────────────────────────────────────────────────────────────
# SECTION 5 — SBV OVERNIGHT / POLICY RATE  (hardcoded, well-known)
# ─────────────────────────────────────────────────────────────────

SBV_RATE_CHANGES = [
    # (effective_date, deposit_rate_pct)   — State Bank of Vietnam 12-month deposit rate cap
    ("2019-01-01", 6.5),
    ("2019-09-19", 6.0),    # SBV cut Sep 2019
    ("2020-03-17", 5.5),    # COVID cut #1
    ("2020-05-13", 5.0),    # COVID cut #2
    ("2020-09-30", 4.5),    # COVID cut #3
    ("2022-09-23", 5.0),    # Rate hike cycle begins
    ("2022-10-25", 6.0),
    ("2022-12-05", 6.5),    # Peak hike
    ("2023-03-15", 6.0),
    ("2023-05-25", 5.5),
    ("2023-06-19", 5.0),
    ("2023-07-11", 4.5),
    ("2023-08-28", 4.0),    # Easing back
    ("2024-01-01", 4.0),
    ("2025-01-01", 4.0),
    ("2025-12-31", 4.0),
]

def build_sbv_rate(date_range):
    changes = [(pd.Timestamp(d), r) for d, r in SBV_RATE_CHANGES]
    rate = []
    for d in date_range:
        r = changes[0][1]
        for dt, val in changes:
            if d >= dt:
                r = val
        rate.append(r)
    return pd.DataFrame({"date": date_range, "sbv_rate": rate})


# ─────────────────────────────────────────────────────────────────
# SECTION 6 — CALENDAR FEATURES  (known-future → TFT advantage)
# ─────────────────────────────────────────────────────────────────

TET_WINDOWS = [
    ("2019-02-04", "2019-02-10"), ("2020-01-23", "2020-01-29"),
    ("2021-02-10", "2021-02-16"), ("2022-01-31", "2022-02-06"),
    ("2023-01-20", "2023-01-26"), ("2024-02-08", "2024-02-14"),
    ("2025-01-27", "2025-02-02"),
]
TET_SET = set()
for s, e in TET_WINDOWS:
    for d in pd.date_range(s, e):
        TET_SET.add(d.date())

def build_calendar(date_range):
    rows = []
    for d in date_range:
        dt = d.date()
        is_tet = int(dt in TET_SET)
        tet_pre = 0
        for s, _ in TET_WINDOWS:
            ws = pd.Timestamp(s).date()
            if 0 < (ws - dt).days <= 14:
                tet_pre = 1
                break
        rows.append({
            "date":              d,
            "cal_day_of_week":   d.dayofweek,
            "cal_month":         d.month,
            "cal_quarter":       d.quarter,
            "cal_day_of_month":  d.day,
            "cal_is_month_end":  int(d.is_month_end),
            "cal_is_qtr_end":    int(d.month in [3,6,9,12] and d.is_month_end),
            "cal_is_tet":        is_tet,
            "cal_is_tet_pre":    tet_pre,
            "cal_rate_month":    int(d.month in [3,6,9,12]),   # SBV typically decides in these
        })
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────
# SECTION 7 — TECHNICAL INDICATORS
# ─────────────────────────────────────────────────────────────────

def add_technicals(df):
    """Compute indicators on a single-ticker DataFrame sorted by date."""
    c = df["close"].copy()
    df = df.copy()

    # Moving averages
    df["ma5"]   = c.rolling(5,  min_periods=1).mean()
    df["ma20"]  = c.rolling(20, min_periods=1).mean()
    df["ma60"]  = c.rolling(60, min_periods=1).mean()

    # EMA / MACD
    df["ema12"] = c.ewm(span=12, adjust=False).mean()
    df["ema26"] = c.ewm(span=26, adjust=False).mean()
    df["macd"]  = df["ema12"] - df["ema26"]
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"]   = df["macd"] - df["macd_signal"]

    # RSI-14
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14, min_periods=1).mean()
    loss  = (-delta.clip(upper=0)).rolling(14, min_periods=1).mean()
    rs    = gain / loss.replace(0, np.nan)
    df["rsi14"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    std20 = c.rolling(20, min_periods=2).std()
    df["bb_upper"] = df["ma20"] + 2 * std20
    df["bb_lower"] = df["ma20"] - 2 * std20
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["ma20"].replace(0, np.nan)
    df["bb_pct"]   = (c - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)

    # ATR-14
    hl  = df["high"] - df["low"]
    hcp = (df["high"] - c.shift(1)).abs()
    lcp = (df["low"]  - c.shift(1)).abs()
    tr  = pd.concat([hl, hcp, lcp], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14, min_periods=1).mean()

    # Log returns at lags
    for lag in [1, 2, 3, 5, 10, 20]:
        df[f"ret_lag{lag}"] = np.log(c / c.shift(lag))

    # Volume features
    df["vol_ma20"]  = df["volume"].rolling(20, min_periods=1).mean()
    df["vol_ratio"] = df["volume"] / df["vol_ma20"].replace(0, np.nan)

    # Price position (close vs 52-week high/low)
    roll252 = c.rolling(252, min_periods=20)
    df["high_52w"] = roll252.max()
    df["low_52w"]  = roll252.min()
    df["pct_from_52w_high"] = (c - df["high_52w"]) / df["high_52w"].replace(0, np.nan)

    return df


# ─────────────────────────────────────────────────────────────────
# SECTION 8 — STATIC COVARIATES  (TFT: fixed per stock)
# ─────────────────────────────────────────────────────────────────

STATIC = {
    #         id  sector  cap  (0=small 1=mid 2=large)
    "HDB": {"stock_id": 0, "static_sector": 1, "static_cap": 1, "static_exchange": 0},
    "TCB": {"stock_id": 1, "static_sector": 1, "static_cap": 2, "static_exchange": 0},
    "TPB": {"stock_id": 2, "static_sector": 1, "static_cap": 0, "static_exchange": 0},
}
# sector 1=banking, exchange 0=HOSE


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*62)
    print("  Stock Dataset Fetcher — HDB · TCB · TPB  (2019–2025)")
    print("="*62)

    # ── 1. Fetch OHLCV for all three stocks ──────────────────────
    print("\n[1/6] Fetching stock prices…")
    stock_frames = []
    for sym in TICKERS:
        time.sleep(0.5)   # polite delay
        df = fetch_stock(sym, START, END)
        stock_frames.append(df)

    all_stocks = pd.concat(stock_frames, ignore_index=True)
    all_stocks["date"] = pd.to_datetime(all_stocks["date"])
    all_stocks = all_stocks.sort_values(["ticker", "date"]).reset_index(drop=True)

    # Get the union of trading dates
    all_dates = pd.DatetimeIndex(sorted(all_stocks["date"].unique()))

    # ── 2. Macro data ─────────────────────────────────────────────
    print("\n[2/6] Fetching macro data…")

    vni = fetch_vnindex(START, END)
    usdvnd = fetch_usdvnd(START, END)
    gold = fetch_gold(START, END)
    sbv = build_sbv_rate(all_dates)

    if usdvnd is None:
        print("  Using USD/VND interpolation fallback.")
        usdvnd = usdvnd_fallback(all_dates)

    # ── 3. Calendar features ──────────────────────────────────────
    print("\n[3/6] Building calendar features…")
    cal = build_calendar(all_dates)

    # ── 4. Merge macro onto date spine ───────────────────────────
    print("\n[4/6] Merging all data sources…")

    macro = (sbv
             .merge(cal, on="date", how="left"))

    if vni is not None:
        vni["date"] = pd.to_datetime(vni["date"])
        macro = macro.merge(vni, on="date", how="left")
    else:
        macro["vni_close"] = np.nan
        macro["vni_change_pct"] = np.nan

    if usdvnd is not None:
        usdvnd["date"] = pd.to_datetime(usdvnd["date"])
        macro = macro.merge(usdvnd, on="date", how="left")
    else:
        macro["usd_vnd"] = np.nan

    if gold is not None:
        gold["date"] = pd.to_datetime(gold["date"])
        macro = macro.merge(gold, on="date", how="left")
    else:
        macro["gold_usd_oz"] = np.nan

    # Convert gold to approximate VND (gold_usd_oz × usd_vnd / 31.1035 = per gram)
    if "gold_usd_oz" in macro.columns and "usd_vnd" in macro.columns:
        macro["gold_vnd_per_gram"] = (
            macro["gold_usd_oz"] * macro["usd_vnd"] / 31.1035
        ).round(0)

    # Forward-fill macro (weekends/holidays in forex/gold but not stocks)
    macro = macro.sort_values("date")
    ffill_cols = ["vni_close","vni_change_pct","usd_vnd","gold_usd_oz","gold_vnd_per_gram"]
    for col in ffill_cols:
        if col in macro.columns:
            macro[col] = macro[col].ffill().bfill()

    # ── 5. Merge stock + macro, compute technicals ───────────────
    print("\n[5/6] Computing technical indicators…")

    tech_frames = []
    for sym in TICKERS:
        s = all_stocks[all_stocks["ticker"] == sym].copy()
        s = add_technicals(s)
        # Add static covariates
        for k, v in STATIC[sym].items():
            s[k] = v
        tech_frames.append(s)

    full = pd.concat(tech_frames, ignore_index=True)
    full = full.merge(macro, on="date", how="left")
    full = full.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── 6. Cross-stock lag prices (TFT advantage) ────────────────
    pivot = full.pivot(index="date", columns="ticker", values="close")
    for src in TICKERS:
        if src in pivot.columns:
            pivot[f"cross_{src}_lag1"] = pivot[src].shift(1)
            pivot[f"cross_{src}_lag2"] = pivot[src].shift(2)

    cross_cols = [c for c in pivot.columns if c.startswith("cross_")]
    cross_df = pivot[cross_cols].reset_index()
    full = full.merge(cross_df, on="date", how="left")

    # ── 7. Train/test split label ─────────────────────────────────
    full["split"] = np.where(full["date"] < SPLIT, "train", "test")

    # ── 8. Final cleanup ──────────────────────────────────────────
    # Round numeric columns sensibly
    int_cols = ["open","high","low","close","volume","vol_ma20"]
    for col in int_cols:
        if col in full.columns:
            full[col] = full[col].round(0).astype("Int64", errors="ignore")

    float_cols = [c for c in full.columns
                  if full[c].dtype == float and c not in int_cols]
    for col in float_cols:
        full[col] = full[col].round(4)

    full = full.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── 9. Save ───────────────────────────────────────────────────
    print("\n[6/6] Saving files…")

    out_full  = "C:\\Users\\PC\\Desktop\\Intern & Thesis\\stock_dataset_full.csv"
    out_train = "C:\\Users\\PC\\Desktop\\Intern & Thesis\\stock_dataset_train.csv"
    out_test  = "C:\\Users\\PC\\Desktop\\Intern & Thesis\\stock_dataset_test.csv"

    full.to_csv(out_full, index=False)
    full[full["split"]=="train"].to_csv(out_train, index=False)
    full[full["split"]=="test" ].to_csv(out_test,  index=False)

    # ── Summary ───────────────────────────────────────────────────
    print("\n" + "="*62)
    print("  Done!")
    print("="*62)
    print(f"\n  Rows total : {len(full):,}")
    print(f"  Rows train : {(full['split']=='train').sum():,}  (<{SPLIT})")
    print(f"  Rows test  : {(full['split']=='test').sum():,}   (>=  {SPLIT})")
    print(f"\n  Output files:")
    for f in [out_full, out_train, out_test]:
        print(f"    {f}")

    print(f"\n  Columns ({len(full.columns)}):")
    groups = {
        "ID / target":    ["date","ticker","split","open","high","low","close","volume"],
        "Technical":      [c for c in full.columns if any(c.startswith(p) for p in
                           ["ma","ema","macd","rsi","bb","atr","ret_lag","vol_ratio",
                            "high_52","low_52","pct_from"])],
        "Macro":          [c for c in full.columns if any(c.startswith(p) for p in
                           ["vni","usd_vnd","gold","sbv"])],
        "Calendar (TFT)": [c for c in full.columns if c.startswith("cal_")],
        "Static   (TFT)": [c for c in full.columns if c.startswith("static_") or c=="stock_id"],
        "Cross-lag (TFT)":[c for c in full.columns if c.startswith("cross_")],
    }
    for grp, cols in groups.items():
        present = [c for c in cols if c in full.columns]
        if present:
            print(f"\n  {grp} ({len(present)}):")
            print(f"    {', '.join(present)}")

    print("\n  Per-ticker date coverage:")
    for sym in TICKERS:
        sub = full[full["ticker"]==sym]
        print(f"    {sym}: {sub['date'].min().date()} → {sub['date'].max().date()}  "
              f"({len(sub):,} rows, last close={sub['close'].iloc[-1]:,})")

    print()
    return full


if __name__ == "__main__":
    df = main()



  Stock Dataset Fetcher — HDB · TCB · TPB  (2019–2025)

[1/6] Fetching stock prices…
  vnstock gave <100 rows for HDB, trying yfinance...
  [yfinance]  HDB.VN: 1748 rows
  vnstock gave <100 rows for TCB, trying yfinance...
  [yfinance]  TCB.VN: 1745 rows


$^VNINDEX.VN: possibly delisted; no price data found  (1d 2019-01-01 -> 2025-12-31)

1 Failed download:
['^VNINDEX.VN']: possibly delisted; no price data found  (1d 2019-01-01 -> 2025-12-31)


  vnstock gave <100 rows for TPB, trying yfinance...
  [yfinance]  TPB.VN: 1748 rows

[2/6] Fetching macro data…


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^VNINDEX"}}}
$^VNINDEX: possibly delisted; no timezone found

1 Failed download:
['^VNINDEX']: possibly delisted; no timezone found


  [yfinance]  VNINDEX: 0 rows
  [yfinance]  USD/VND: 1821 rows
  [yfinance]  Gold (GC=F): 1761 rows

[3/6] Building calendar features…

[4/6] Merging all data sources…

[5/6] Computing technical indicators…

[6/6] Saving files…

  Done!

  Rows total : 5,241
  Rows train : 5,046  (<2025-10-01)
  Rows test  : 195   (>=  2025-10-01)

  Output files:
    C:\Users\PC\Desktop\Intern & Thesis\stock_dataset_full.csv
    C:\Users\PC\Desktop\Intern & Thesis\stock_dataset_train.csv
    C:\Users\PC\Desktop\Intern & Thesis\stock_dataset_test.csv

  Columns (58):

  ID / target (8):
    date, ticker, split, open, high, low, close, volume

  Technical (24):
    ma5, ma20, ma60, ema12, ema26, macd, macd_signal, macd_hist, rsi14, bb_upper, bb_lower, bb_width, bb_pct, atr14, ret_lag1, ret_lag2, ret_lag3, ret_lag5, ret_lag10, ret_lag20, vol_ratio, high_52w, low_52w, pct_from_52w_high

  Macro (6):
    sbv_rate, vni_close, vni_change_pct, usd_vnd, gold_usd_oz, gold_vnd_per_gram

  Calendar (TFT) (9):
   